In [2]:
import pandas as pd
import pyreadstat

In [4]:
df = pd.read_csv("../data/raw/hl_combined.csv", low_memory=False)
print(f"Rows: {df.shape[0]:,} | Columns: {df.shape[1]}")

Rows: 479,999 | Columns: 115


In [7]:
df.head()

,HH1,HH2,HL1,HL3,HL4,HL5M,HL5Y,HL6,HL7,HL7A,...,windex10u,wscorer,windex5r,windex10r,psu,stratum,province,ED10D,insurance,disability2
0,1.0,1.0,1.0,1.0,1.0,98.0,1980.0,39.0,1.0,1.0,...,NaN,-0.073337,3.0,5.0,1.0,33.0,Balochistan,NaN,NaN,NaN
1,1.0,1.0,2.0,2.0,2.0,98.0,1981.0,38.0,1.0,1.0,...,NaN,-0.073337,3.0,5.0,1.0,33.0,Balochistan,NaN,NaN,NaN
2,1.0,1.0,3.0,3.0,2.0,4.0,2002.0,17.0,1.0,1.0,...,NaN,-0.073337,3.0,5.0,1.0,33.0,Balochistan,NaN,NaN,NaN
3,1.0,1.0,4.0,3.0,1.0,3.0,2003.0,16.0,1.0,1.0,...,NaN,-0.073337,3.0,5.0,1.0,33.0,Balochistan,NaN,NaN,NaN
4,1.0,1.0,5.0,3.0,2.0,5.0,2005.0,14.0,1.0,1.0,...,NaN,-0.073337,3.0,5.0,1.0,33.0,Balochistan,NaN,NaN,NaN


In [8]:
cols_to_keep = [
    'HH1', 'HH2',           # Merge keys
    'schage',               # not sure what this is
    'HL4',                  # Child sex
    # 'BR2',                  # Birth weight -- not there
    # 'BR3',                  # Breastfeeding -- not there
    # 'HAZ', 'WAZ', 'WHZ',    # Z scores -- not there
    # 'HAZ2', 'WAZ2', 'WHZ2', # Stunted/Underweight/Wasted -- not there
    'melevel',              # Mother education
    'HH6',                  # Urban/Rural
    'HH7',                  # District
    'windex5',              # Wealth index
    'disability',           # Disability
    'province'              # Province
]

In [17]:
# select desired columns and create a dataframe from them
hl_clean = df[cols_to_keep].copy()

# rename columns for convenience
hl_clean.columns = [
    'cluster_id', 'household_id',
    'child_age_months', 'child_sex',
    'mother_education', 'urban_rural',
    'district', 'wealth_index',
    'disability', 'province'
]

In [18]:
print(f"Rows: {hl_clean.shape[0]:,} | Columns: {hl_clean.shape[1]}")
print(hl_clean.isnull().sum())

Rows: 479,999 | Columns: 10
cluster_id               0
household_id             0
child_age_months         0
child_sex                0
mother_education    250045
urban_rural              0
district                 0
wealth_index             0
disability          204488
province                 0
dtype: int64


In [19]:
# disability — fill with 0
hl_clean['disability'] = hl_clean['disability'].fillna(0)

# mother_education — fill with mode
hl_clean['mother_education'] = hl_clean['mother_education'].fillna(
    hl_clean['mother_education'].mode()[0])

In [20]:
# Encode child_sex: 1=male→0, 2=female→1
hl_clean['child_sex'] = hl_clean['child_sex'].map({1.0: 0, 2.0: 1})

# Encode urban_rural: 1=urban→0, 2=rural→1
hl_clean['urban_rural'] = hl_clean['urban_rural'].map({1.0: 0, 2.0: 1})

# Encode province
hl_clean['province'] = hl_clean['province'].map({
    'Balochistan': 0,
    'KPK': 1,
    'Sindh': 2
})

In [22]:
print(f"Final shape: {hl_clean.shape[0]:,} rows | {hl_clean.shape[1]} columns")
# print(f"\nStunting rate: {hl_clean['stunted'].mean()*100:.1f}%")
print(f"\nMissing values:")
print(hl_clean.isnull().sum())
print(f"\nColumns: {hl_clean.columns.tolist()}")

Final shape: 479,999 rows | 10 columns

Missing values:
cluster_id          0
household_id        0
child_age_months    0
child_sex           0
mother_education    0
urban_rural         0
district            0
wealth_index        0
disability          0
province            0
dtype: int64

Columns: ['cluster_id', 'household_id', 'child_age_months', 'child_sex', 'mother_education', 'urban_rural', 'district', 'wealth_index', 'disability', 'province']


In [24]:
hl_clean.to_csv("../data/processed/hl_cleaned.csv", index=False)
print("\nSaved!")


Saved!
